# Diffusion Models for High-Resolution Image Generation & Reconstruction
### DDPM Implementation using PyTorch on CelebA-HQ Dataset

## 1. Environment Setup & Dataset Check

In [ ]:
import os
from pathlib import Path

# Check Kaggle input directory for datasets
input_dir = Path('/kaggle/input')
image_exts = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}


def count_images(root: Path, limit=None):
    """Count image files recursively under root."""
    count = 0
    for p in root.rglob('*'):
        if p.is_file() and p.suffix.lower() in image_exts:
            count += 1
            if limit is not None and count >= limit:
                return count
    return count


print('Datasets found in /kaggle/input:')
dataset_roots = []
for d in sorted(input_dir.iterdir()):
    if not d.is_dir():
        continue
    total_imgs = count_images(d)
    dataset_roots.append((d, total_imgs))
    print(f'- {d.name}: {total_imgs} images')

# Optional: set this manually if you want a specific dataset.
# Examples based on your screenshots:
#   /kaggle/input/celebahq-256-images-only
#   /kaggle/input/wikiart
#   /kaggle/input/ffhq-face-data-set
FORCE_DATASET = None

if FORCE_DATASET:
    DATA_DIR = FORCE_DATASET
else:
    non_empty = [(p, n) for p, n in dataset_roots if n > 0]
    DATA_DIR = str(max(non_empty, key=lambda x: x[1])[0]) if non_empty else None

print('Selected DATA_DIR:', DATA_DIR)
if DATA_DIR is None:
    raise ValueError('No images found in /kaggle/input. Check dataset attachment and path.')

## 2. Data Preprocessing

In [ ]:
import os
import random
import torch
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader, Subset
from torch.utils.data._utils.collate import default_collate
from PIL import Image, ImageFile, UnidentifiedImageError

# Allow loading slightly damaged images instead of crashing.
ImageFile.LOAD_TRUNCATED_IMAGES = True

IMG_SIZE = 128
BATCH_SIZE = 16

# Time-budget controls (based on your measured speed).
TARGET_TOTAL_HOURS = 2.0
PLANNED_EPOCHS = 30
MEASURED_IT_PER_SEC = 1.25
SAFETY_FACTOR = 0.85  # keep margin for logging/checkpoint overhead


def is_valid_image(path):
    """Fast integrity check for image files."""
    try:
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


class ImageDataset(Dataset):
    def __init__(self, folder, transform=None):
        if folder is None:
            raise ValueError('DATA_DIR is None. Set a valid dataset path.')

        exts = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
        raw_files = []

        # Recursive scan so nested structures work:
        # - celebahq256_imgs/train/*.jpg
        # - wikiart/wikiart/<style>/*.jpg
        # - FFHQ/thumbnails128×128/*.png
        for root, _, files in os.walk(folder):
            for f in files:
                if f.lower().endswith(exts):
                    raw_files.append(os.path.join(root, f))

        raw_files.sort()
        self.transform = transform

        # Remove corrupted files once, so workers don't crash during training.
        self.files = [p for p in raw_files if is_valid_image(p)]
        skipped = len(raw_files) - len(self.files)

        if len(self.files) == 0:
            raise ValueError(
                f'No valid image files found under {folder}. '
                'Check DATA_DIR or file integrity.'
            )

        print(f'Valid images: {len(self.files)} | Skipped corrupted: {skipped}')

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        # Return None on rare runtime decode failures; collate_fn will drop it.
        try:
            path = self.files[idx]
            img = Image.open(path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            return img
        except (OSError, UnidentifiedImageError, ValueError):
            return None


def safe_collate(batch):
    batch = [x for x in batch if x is not None]
    if len(batch) == 0:
        return None
    return default_collate(batch)


transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5])
])

full_dataset = ImageDataset(DATA_DIR, transform)

# Compute target steps/epoch and target sample count to fit 2-hour total budget.
seconds_per_epoch_budget = (TARGET_TOTAL_HOURS * 3600.0) / PLANNED_EPOCHS
TARGET_STEPS_PER_EPOCH = max(1, int(seconds_per_epoch_budget * MEASURED_IT_PER_SEC * SAFETY_FACTOR))
target_samples = TARGET_STEPS_PER_EPOCH * BATCH_SIZE

if len(full_dataset) > target_samples:
    rng = random.Random(42)
    indices = list(range(len(full_dataset)))
    rng.shuffle(indices)
    indices = indices[:target_samples]
    dataset = Subset(full_dataset, indices)
else:
    dataset = full_dataset

# Fewer workers are often more stable in Kaggle with heavy image decode.
NUM_WORKERS = 2

dataloader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    collate_fn=safe_collate,
    drop_last=True
)

print('Using reduced dataset for time budget.')
print('Planned epochs:', PLANNED_EPOCHS)
print('Target total hours:', TARGET_TOTAL_HOURS)
print('Measured it/s:', MEASURED_IT_PER_SEC)
print('Target steps/epoch:', TARGET_STEPS_PER_EPOCH)
print('Target sample count:', target_samples)
print('Actual samples used:', len(dataset))
print('Actual batches/epoch:', len(dataloader))

## 3. Forward Diffusion Process

In [ ]:
import torch
import matplotlib.pyplot as plt

T = 300  # timesteps

def linear_schedule(T):
    beta_start = 1e-4
    beta_end = 0.02
    return torch.linspace(beta_start, beta_end, T)

betas = linear_schedule(T)
alphas = 1.0 - betas
alpha_cumprod = torch.cumprod(alphas, dim=0)
sqrt_alpha_cumprod = torch.sqrt(alpha_cumprod)
sqrt_one_minus_alpha_cumprod = torch.sqrt(1.0 - alpha_cumprod)

def q_sample(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    s1 = sqrt_alpha_cumprod[t].view(-1, 1, 1, 1).to(x0.device)
    s2 = sqrt_one_minus_alpha_cumprod[t].view(-1, 1, 1, 1).to(x0.device)
    return s1 * x0 + s2 * noise, noise

# Visualize 5 forward diffusion steps
sample_img = next(iter(dataloader))[0].unsqueeze(0)  # single image
steps = [0, 60, 120, 180, 299]

fig, axes = plt.subplots(1, len(steps), figsize=(15, 3))
for i, step in enumerate(steps):
    t = torch.tensor([step])
    noisy, _ = q_sample(sample_img, t)
    img_np = noisy[0].permute(1, 2, 0).numpy()
    img_np = (img_np * 0.5 + 0.5).clip(0, 1)
    axes[i].imshow(img_np)
    axes[i].set_title(f't={step}')
    axes[i].axis('off')
plt.suptitle('Forward Diffusion Steps')
plt.tight_layout()
plt.savefig('/kaggle/working/forward_diffusion.png', dpi=100)
plt.show()

## 4. U-Net Architecture

In [ ]:
import torch
import torch.nn as nn
import math

class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(
            nn.Linear(dim, dim * 4),
            nn.SiLU(),
            nn.Linear(dim * 4, dim)
        )

    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = t[:, None].float() * freqs[None]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        return self.mlp(emb)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1 = nn.Sequential(nn.GroupNorm(8, in_ch), nn.SiLU(), nn.Conv2d(in_ch, out_ch, 3, padding=1))
        self.time_proj = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, out_ch))
        self.conv2 = nn.Sequential(nn.GroupNorm(8, out_ch), nn.SiLU(), nn.Conv2d(out_ch, out_ch, 3, padding=1))
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t_emb):
        h = self.conv1(x)
        h = h + self.time_proj(t_emb)[:, :, None, None]
        h = self.conv2(h)
        return h + self.skip(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.res = ResBlock(in_ch, out_ch, time_dim)
        self.down = nn.Conv2d(out_ch, out_ch, 4, stride=2, padding=1)

    def forward(self, x, t):
        x = self.res(x, t)
        return self.down(x), x

class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, time_dim):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch, 4, stride=2, padding=1)
        self.res = ResBlock(in_ch + skip_ch, out_ch, time_dim)

    def forward(self, x, skip, t):
        x = self.up(x)
        x = torch.cat([x, skip], dim=1)
        return self.res(x, t)

class UNet(nn.Module):
    def __init__(self, in_ch=3, base_ch=64, time_dim=256):
        super().__init__()
        self.time_emb = TimeEmbedding(time_dim)
        self.init_conv = nn.Conv2d(in_ch, base_ch, 3, padding=1)

        self.down1 = Down(base_ch, base_ch * 2, time_dim)
        self.down2 = Down(base_ch * 2, base_ch * 4, time_dim)

        self.mid = ResBlock(base_ch * 4, base_ch * 4, time_dim)

        self.up1 = Up(base_ch * 4, base_ch * 4, base_ch * 2, time_dim)
        self.up2 = Up(base_ch * 2, base_ch * 2, base_ch, time_dim)

        self.out = nn.Sequential(nn.GroupNorm(8, base_ch), nn.SiLU(), nn.Conv2d(base_ch, in_ch, 1))

    def forward(self, x, t):
        t_emb = self.time_emb(t)
        x = self.init_conv(x)
        x, s1 = self.down1(x, t_emb)
        x, s2 = self.down2(x, t_emb)
        x = self.mid(x, t_emb)
        x = self.up1(x, s2, t_emb)
        x = self.up2(x, s1, t_emb)
        return self.out(x)


def unwrap_model(m):
    return m.module if isinstance(m, nn.DataParallel) else m


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0

if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

print('Device:', device)
print('GPU count:', gpu_count)
if gpu_count > 0:
    for i in range(gpu_count):
        print(f'GPU {i}: {torch.cuda.get_device_name(i)}')

model = UNet()
if gpu_count > 1:
    print('Using multi-GPU training with nn.DataParallel')
    model = nn.DataParallel(model)

model = model.to(device)
total_params = sum(p.numel() for p in unwrap_model(model).parameters())
print('Parameters:', total_params)

## 5. Training

In [ ]:
import torch
import torch.nn.functional as F
from torch.amp import GradScaler, autocast
from tqdm.auto import tqdm
import time

EPOCHS = 30
LR = 2e-4

optimizer = torch.optim.AdamW(model.parameters(), lr=LR)
use_amp = (device.type == 'cuda')
scaler = GradScaler('cuda', enabled=use_amp)

# move noise schedule to device
sqrt_alpha_cumprod = sqrt_alpha_cumprod.to(device)
sqrt_one_minus_alpha_cumprod = sqrt_one_minus_alpha_cumprod.to(device)

def q_sample_device(x0, t, noise=None):
    if noise is None:
        noise = torch.randn_like(x0)
    s1 = sqrt_alpha_cumprod[t].view(-1, 1, 1, 1)
    s2 = sqrt_one_minus_alpha_cumprod[t].view(-1, 1, 1, 1)
    return s1 * x0 + s2 * noise, noise

def format_time(seconds):
    seconds = int(max(0, seconds))
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    if h > 0:
        return f'{h:02d}:{m:02d}:{s:02d}'
    return f'{m:02d}:{s:02d}'

# Respect target steps from preprocessing cell if present.
max_steps_per_epoch = globals().get('TARGET_STEPS_PER_EPOCH', len(dataloader))

loss_history = []
train_start = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    num_batches = 0
    epoch_start = time.time()

    pbar = tqdm(dataloader, total=min(len(dataloader), max_steps_per_epoch), desc=f'Epoch {epoch}/{EPOCHS}', leave=True)
    for step, imgs in enumerate(pbar, start=1):
        if step > max_steps_per_epoch:
            break

        if imgs is None:
            continue

        imgs = imgs.to(device, non_blocking=True)
        t = torch.randint(0, T, (imgs.size(0),), device=device)
        noise = torch.randn_like(imgs)
        noisy_imgs, noise = q_sample_device(imgs, t, noise)

        optimizer.zero_grad(set_to_none=True)
        with autocast(device_type='cuda', enabled=use_amp):
            pred = model(noisy_imgs, t)
            loss = F.mse_loss(pred, noise)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        num_batches += 1

        avg_so_far = total_loss / max(1, num_batches)
        pbar.set_postfix(loss=f'{loss.item():.4f}', avg=f'{avg_so_far:.4f}')

    if num_batches == 0:
        raise RuntimeError('All batches were empty after filtering corrupted images.')

    avg = total_loss / num_batches
    loss_history.append(avg)

    epoch_time = time.time() - epoch_start
    elapsed_total = time.time() - train_start
    avg_epoch_time = elapsed_total / epoch
    eta_total = avg_epoch_time * (EPOCHS - epoch)

    print(
        f'Epoch {epoch}/{EPOCHS} complete | '
        f'Steps: {num_batches}/{max_steps_per_epoch} | '
        f'Loss: {avg:.4f} | '
        f'Epoch Time: {format_time(epoch_time)} | '
        f'ETA Remaining: {format_time(eta_total)}'
    )

model_to_save = model.module if isinstance(model, torch.nn.DataParallel) else model
torch.save(model_to_save.state_dict(), '/kaggle/working/ddpm_model.pth')
print('Model saved.')

## 6. Loss Plot

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(loss_history) + 1), loss_history)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('Training Loss')
plt.tight_layout()
plt.savefig('/kaggle/working/loss_plot.png', dpi=100)
plt.show()

## 7. Reverse Diffusion (Sampling)

In [ ]:
import torch

betas_d = betas.to(device)
alphas_d = alphas.to(device)
alpha_cumprod_d = alpha_cumprod.to(device)
alpha_cumprod_prev = torch.cat([torch.tensor([1.0], device=device), alpha_cumprod_d[:-1]])

@torch.no_grad()
def p_sample(model, x, t_idx):
    model.eval()
    t_tensor = torch.full((x.size(0),), t_idx, device=device, dtype=torch.long)
    pred_noise = model(x, t_tensor)

    beta = betas_d[t_idx]
    alpha = alphas_d[t_idx]
    acp = alpha_cumprod_d[t_idx]

    mean = (1 / alpha.sqrt()) * (x - (beta / (1 - acp).sqrt()) * pred_noise)
    if t_idx == 0:
        return mean
    noise = torch.randn_like(x)
    var = beta * (1 - alpha_cumprod_prev[t_idx]) / (1 - acp)
    return mean + var.sqrt() * noise

@torch.no_grad()
def generate(model, n=5, save_steps=True):
    model.eval()
    x = torch.randn(n, 3, IMG_SIZE, IMG_SIZE, device=device)
    intermediates = []
    steps_to_save = [T-1, T//4*3, T//2, T//4, 0]
    for t in reversed(range(T)):
        x = p_sample(model, x, t)
        if save_steps and t in steps_to_save:
            intermediates.append(x.clone().cpu())
    return x.cpu(), intermediates

print('Generating images...')
generated, intermediates = generate(model, n=5)
print('Generation done.')

## 8. Visualize Generated Images

In [ ]:
import matplotlib.pyplot as plt

def show_images(imgs, title, fname):
    fig, axes = plt.subplots(1, len(imgs), figsize=(3 * len(imgs), 3))
    if len(imgs) == 1:
        axes = [axes]
    for ax, img in zip(axes, imgs):
        img_np = img.permute(1, 2, 0).numpy()
        img_np = (img_np * 0.5 + 0.5).clip(0, 1)
        ax.imshow(img_np)
        ax.axis('off')
    plt.suptitle(title)
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/{fname}', dpi=100)
    plt.show()

show_images([generated[i] for i in range(5)], 'Generated Images', 'generated_images.png')

# Show reverse diffusion steps for 1st image
step_imgs = [s[0] for s in intermediates]
show_images(step_imgs, 'Reverse Diffusion Steps', 'reverse_steps.png')

## 9. Image Reconstruction Task

In [ ]:
import torch
import matplotlib.pyplot as plt

# Pick a target image from dataset
target = next(iter(dataloader))[0:1].to(device)

# Add noise at t=250 then denoise back
t_noisy = torch.tensor([250], device=device)
noised, _ = q_sample_device(target, t_noisy)

x = noised.clone()
for t in reversed(range(251)):
    x = p_sample(model, x, t)

reconstructed = x.cpu()

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, img, title in zip(axes,
                           [target.cpu()[0], noised.cpu()[0], reconstructed[0]],
                           ['Original', 'Noised (t=250)', 'Reconstructed']):
    np_img = img.permute(1, 2, 0).numpy()
    np_img = (np_img * 0.5 + 0.5).clip(0, 1)
    ax.imshow(np_img)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()
plt.savefig('/kaggle/working/reconstruction.png', dpi=100)
plt.show()

## 10. Quantitative Evaluation (PSNR & SSIM)

In [ ]:
import numpy as np
from skimage.metrics import peak_signal_noise_ratio as psnr, structural_similarity as ssim

def to_np(t):
    img = t.squeeze().permute(1, 2, 0).numpy()
    return (img * 0.5 + 0.5).clip(0, 1)

orig_np = to_np(target.cpu())
recon_np = to_np(reconstructed)

psnr_val = psnr(orig_np, recon_np, data_range=1.0)
ssim_val = ssim(orig_np, recon_np, data_range=1.0, channel_axis=2)

print(f'PSNR: {psnr_val:.2f} dB')
print(f'SSIM: {ssim_val:.4f}')

## 11. Gradio App Deployment

In [ ]:
!pip install gradio -q

In [ ]:
import gradio as gr
import torch
import numpy as np
from PIL import Image

def run_generation():
    model.eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    steps_out = []
    checkpoints = set([T - 1, T // 4 * 3, T // 2, T // 4, 0])

    with torch.no_grad():
        for t in reversed(range(T)):
            x = p_sample(model, x, t)
            if t in checkpoints:
                img = x[0].cpu().permute(1, 2, 0).numpy()
                img = (img * 0.5 + 0.5).clip(0, 1)
                steps_out.append((img * 255).astype(np.uint8))

    final = steps_out[-1]
    gallery = [Image.fromarray(s) for s in steps_out]
    return gallery, Image.fromarray(final)

with gr.Blocks(title='DDPM Image Generator') as demo:
    gr.Markdown('## DDPM - Diffusion Model Image Generator')
    btn = gr.Button('Generate Image from Noise')
    with gr.Row():
        steps_gallery = gr.Gallery(label='Denoising Steps')
        final_img = gr.Image(label='Final Generated Image')
    btn.click(fn=run_generation, inputs=[], outputs=[steps_gallery, final_img])

demo.launch(share=True)

# Save the app script for deployment
app_code = '''
import gradio as gr
import torch
import numpy as np
from PIL import Image
import math
import torch.nn as nn

T = 300
IMG_SIZE = 128
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

betas = torch.linspace(1e-4, 0.02, T).to(device)
alphas = 1.0 - betas
alpha_cumprod = torch.cumprod(alphas, dim=0)
alpha_cumprod_prev = torch.cat([torch.tensor([1.0], device=device), alpha_cumprod[:-1]])

class TimeEmbedding(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        self.mlp = nn.Sequential(nn.Linear(dim, dim*4), nn.SiLU(), nn.Linear(dim*4, dim))
    def forward(self, t):
        half = self.dim // 2
        freqs = torch.exp(-math.log(10000) * torch.arange(half, device=t.device) / half)
        args = t[:, None].float() * freqs[None]
        emb = torch.cat([torch.sin(args), torch.cos(args)], dim=-1)
        return self.mlp(emb)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.conv1 = nn.Sequential(nn.GroupNorm(8, in_ch), nn.SiLU(), nn.Conv2d(in_ch, out_ch, 3, padding=1))
        self.time_proj = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, out_ch))
        self.conv2 = nn.Sequential(nn.GroupNorm(8, out_ch), nn.SiLU(), nn.Conv2d(out_ch, out_ch, 3, padding=1))
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()
    def forward(self, x, t_emb):
        h = self.conv1(x) + self.time_proj(t_emb)[:, :, None, None]
        return self.conv2(h) + self.skip(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim):
        super().__init__()
        self.res = ResBlock(in_ch, out_ch, time_dim)
        self.down = nn.Conv2d(out_ch, out_ch, 4, stride=2, padding=1)
    def forward(self, x, t):
        x = self.res(x, t)
        return self.down(x), x

class Up(nn.Module):
    def __init__(self, in_ch, skip_ch, out_ch, time_dim):
        super().__init__()
        self.up = nn.ConvTranspose2d(in_ch, in_ch, 4, stride=2, padding=1)
        self.res = ResBlock(in_ch + skip_ch, out_ch, time_dim)
    def forward(self, x, skip, t):
        x = torch.cat([self.up(x), skip], dim=1)
        return self.res(x, t)

class UNet(nn.Module):
    def __init__(self, in_ch=3, base_ch=64, time_dim=256):
        super().__init__()
        self.time_emb = TimeEmbedding(time_dim)
        self.init_conv = nn.Conv2d(in_ch, base_ch, 3, padding=1)
        self.down1 = Down(base_ch, base_ch*2, time_dim)
        self.down2 = Down(base_ch*2, base_ch*4, time_dim)
        self.mid = ResBlock(base_ch*4, base_ch*4, time_dim)
        self.up1 = Up(base_ch*4, base_ch*4, base_ch*2, time_dim)
        self.up2 = Up(base_ch*2, base_ch*2, base_ch, time_dim)
        self.out = nn.Sequential(nn.GroupNorm(8, base_ch), nn.SiLU(), nn.Conv2d(base_ch, in_ch, 1))
    def forward(self, x, t):
        t_emb = self.time_emb(t)
        x = self.init_conv(x)
        x, s1 = self.down1(x, t_emb)
        x, s2 = self.down2(x, t_emb)
        x = self.mid(x, t_emb)
        x = self.up1(x, s2, t_emb)
        x = self.up2(x, s1, t_emb)
        return self.out(x)

model = UNet().to(device)
model.load_state_dict(torch.load('ddpm_model.pth', map_location=device))
model.eval()

def p_sample(x, t_idx):
    t_tensor = torch.full((x.size(0),), t_idx, device=device, dtype=torch.long)
    pred_noise = model(x, t_tensor)
    beta = betas[t_idx]
    alpha = alphas[t_idx]
    acp = alpha_cumprod[t_idx]
    mean = (1 / alpha.sqrt()) * (x - (beta / (1 - acp).sqrt()) * pred_noise)
    if t_idx == 0:
        return mean
    return mean + (beta * (1 - alpha_cumprod_prev[t_idx]) / (1 - acp)).sqrt() * torch.randn_like(x)

def generate():
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
    steps_out = []
    checkpoints = {T-1, T//4*3, T//2, T//4, 0}
    with torch.no_grad():
        for t in reversed(range(T)):
            x = p_sample(x, t)
            if t in checkpoints:
                img = x[0].cpu().permute(1,2,0).numpy()
                img = (img * 0.5 + 0.5).clip(0, 1)
                steps_out.append(Image.fromarray((img*255).astype(np.uint8)))
    return steps_out, steps_out[-1]

with gr.Blocks(title="DDPM") as demo:
    gr.Markdown("## DDPM - Generate Images from Noise")
    btn = gr.Button("Generate")
    with gr.Row():
        gallery = gr.Gallery(label="Denoising Steps")
        final = gr.Image(label="Final Image")
    btn.click(generate, [], [gallery, final])

demo.launch()
'''

with open('/kaggle/working/app.py', 'w') as f:
    f.write(app_code)
print('app.py saved.')

## 12. Save All Files

In [ ]:
import os

files = os.listdir('/kaggle/working')
print('Files saved in /kaggle/working:')
for f in sorted(files):
    size = os.path.getsize(f'/kaggle/working/{f}')
    print(f'  {f}  ({size/1024:.1f} KB)')